In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, BaggingClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.metrics import accuracy_score
import pickle

# ----------------------------
# Load dataset
# ----------------------------
data = pd.read_csv("crop_recommendation.csv")

# Encode crop labels
crop_dict = {
    'rice': 1, 'maize': 2, 'jute': 3, 'cotton': 4, 'coconut': 5, 'papaya': 6, 'orange': 7,
    'apple': 8, 'muskmelon': 9, 'watermelon': 10, 'grapes': 11, 'mango': 12, 'banana': 13, 'pomegranate': 14,
    'lentil': 15, 'blackgram': 16, 'mungbean': 17, 'mothbeans': 18, 'pigeonpeas': 19, 'kidneybeans': 20,
    'chickpea': 21, 'coffee': 22
}
reverse_crop_dict = {v: k.capitalize() for k, v in crop_dict.items()}

data['crop_num'] = data['label'].map(crop_dict)
data.drop(['label'], axis=1, inplace=True)

# ----------------------------
# Train-test split
# ----------------------------
X = data.drop(['crop_num'], axis=1)
y = data['crop_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
ms = MinMaxScaler()
X_train = ms.fit_transform(X_train)
X_test = ms.transform(X_test)

# ----------------------------
# Train and compare models
# ----------------------------
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Bagging': BaggingClassifier(),
    'AdaBoost': AdaBoostClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'Extra Trees': ExtraTreeClassifier(),
}

print("🔍 Model Performance (Training vs Testing Accuracy)\n")
for name, model in models.items():
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    print(f"{name}")
    print(f"  Training Accuracy : {train_acc:.4f}")
    print(f"  Testing Accuracy  : {test_acc:.4f}")
    print("---------------------------------------------------")

# ----------------------------
# Final model (Random Forest)
# ----------------------------
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)

# ----------------------------
# Recommendation function
# ----------------------------
def predict(N, P, K, temperature, humidity, ph, rainfall):
    features = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall'])
    transformed_features = ms.transform(features)
    prediction = rfc.predict(transformed_features)
    return reverse_crop_dict[prediction[0]]

# # ----------------------------
# # Example usage
# # ----------------------------
# if __name__ == "__main__":
#     N = int(input("Enter value of N: "))
#     P = int(input("Enter value of P: "))
#     K = int(input("Enter value of K: "))
#     temperature = float(input("Enter value of temperature: "))
#     humidity = float(input("Enter value of humidity: "))
#     ph = float(input("Enter value of pH: "))   # ✅ allows integer or float
#     rainfall = float(input("Enter value of rainfall: "))

#     best_crop = recommendation(N, P, K, temperature, humidity, ph, rainfall)
#     print(f"\n🌱 Recommended Crop: {best_crop}")

import pickle

bundle = {
    "model": rfc,
    "scaler": ms
}

with open("crop_recommendation.pkl", "wb") as f:
    pickle.dump(bundle, f)

🔍 Model Performance (Training vs Testing Accuracy)

Logistic Regression
  Training Accuracy : 0.9477
  Testing Accuracy  : 0.9182
---------------------------------------------------
Naive Bayes
  Training Accuracy : 0.9949
  Testing Accuracy  : 0.9955
---------------------------------------------------
Support Vector Machine
  Training Accuracy : 0.9852
  Testing Accuracy  : 0.9682
---------------------------------------------------
K-Nearest Neighbors
  Training Accuracy : 0.9909
  Testing Accuracy  : 0.9705
---------------------------------------------------
Decision Tree
  Training Accuracy : 1.0000
  Testing Accuracy  : 0.9841
---------------------------------------------------
Random Forest
  Training Accuracy : 1.0000
  Testing Accuracy  : 0.9932
---------------------------------------------------
Bagging
  Training Accuracy : 0.9994
  Testing Accuracy  : 0.9909
---------------------------------------------------
AdaBoost
  Training Accuracy : 0.1341
  Testing Accuracy  : 0.1455


In [6]:
# Load the saved bundle
with open("crop_recommendation.pkl", "rb") as f:
    bundle = pickle.load(f)

loaded_model = bundle["model"]
loaded_scaler = bundle["scaler"]

# Update the predict function to use the loaded model and scaler
def predict_loaded(N, P, K, temperature, humidity, ph, rainfall):
    features = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall'])
    transformed_features = loaded_scaler.transform(features)
    prediction = loaded_model.predict(transformed_features)
    return reverse_crop_dict[prediction[0]]

# Test the prediction function with example values
N = 30
P = 42
K = 43
temperature = 35.879744
humidity = 42.002744
ph = 7.02985
rainfall = 402.935536

recommended_crop = predict_loaded(N, P, K, temperature, humidity, ph, rainfall)
print(f"Recommended Crop: {recommended_crop}")

Recommended Crop: Mango
